# Decision-tree paths on synthetic tabular data

Fit a shallow scikit-learn decision tree and explain the exact branch predicates for one held-out row. The path is a faithful description of this tree's computation, not a causal explanation, SHAP decomposition, or evidence that correlated features are irrelevant. Impurity-based importance is reported separately and can be biased. Reference: [scikit-learn decision-tree guide](https://scikit-learn.org/stable/modules/tree.html).

Offline CPU tutorial. Fixed seeds and two PyTorch threads; small synthetic data only.


In [1]:
import os, sys, time, resource, json
from pathlib import Path
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "src"))
import numpy as np
import torch
from torch import nn
torch.set_num_threads(2)
torch.set_num_interop_threads(2)
torch.manual_seed(7)
np.random.seed(7)
started = time.perf_counter()
print("CPU only; torch", torch.__version__, "threads", torch.get_num_threads())



CPU only; torch 2.10.0.dev20251025+cpu threads 2


In [2]:
from sklearn.tree import DecisionTreeClassifier
from autoexplain.tabular import explain_tree_path
rng = np.random.default_rng(7)
features = rng.normal(size=(160, 3))
labels = ((features[:, 0] > 0) & (features[:, 1] < 0.5)).astype(int)
model = DecisionTreeClassifier(max_depth=3, random_state=7)
model.fit(features[:120], labels[:120])
print("Held-out accuracy:", model.score(features[120:], labels[120:]))
row = features[123]
explanation = explain_tree_path(model, row, ["signal_a", "signal_b", "noise"])
print(json.dumps(explanation, indent=2))
assert explanation["leaf"] == int(model.apply(row[None])[0])
assert explanation["prediction"] == int(model.predict(row[None])[0])
print("Impurity importance (not local attribution):", model.feature_importances_.tolist())
for step in explanation["path"]:
    assert (step["value"] <= step["threshold"]) == (step["operator"] == "<=")



Held-out accuracy: 1.0
{
  "path": [
    {
      "node": 0,
      "feature": "signal_a",
      "feature_index": 0,
      "value": 0.7062350511550903,
      "operator": ">",
      "threshold": -0.003064066346269101
    },
    {
      "node": 2,
      "feature": "signal_b",
      "feature_index": 1,
      "value": -0.14993900060653687,
      "operator": "<=",
      "threshold": 0.5232496857643127
    }
  ],
  "leaf": 3,
  "prediction": 1,
  "classes": [
    0,
    1
  ],
  "probabilities": [
    0.0,
    1.0
  ]
}
Impurity importance (not local attribution): [0.6656534954407295, 0.3343465045592705, 0.0]


In [3]:
elapsed = time.perf_counter() - started
rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
# Linux reports KiB; macOS reports bytes. This is this kernel's lifetime high-water mark.
peak_mib = rss / (1024 * 1024 if sys.platform == "darwin" else 1024)
print(json.dumps({"tutorial_compute_wall_seconds": round(elapsed, 3),
                  "kernel_lifetime_peak_rss_mib": round(peak_mib, 2),
                  "memory_scope": "kernel process only; includes imports; excludes child processes"}))



{"tutorial_compute_wall_seconds": 0.519, "kernel_lifetime_peak_rss_mib": 393.7, "memory_scope": "kernel process only; includes imports; excludes child processes"}
